# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 colorectal cancer survivors dataset (77 cases, rich clinical and molecular features) described by a Croissant schema and accessible via `mlcroissant`. All fields, columns, and record sets are referenced explicitly by their `@id` for maximum reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant


## 1. Data Loading

Let's load the metadata and records of the dataset with `mlcroissant`. We'll display the dataset's title and detailed description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:\n", metadata.description)


## 2. Data Overview

Explore available record sets (tables) and their schemas. We'll list all `@id`s and titles of top-level record sets/fields for reference.

**Note:** Throughout this exploration, all dataset entities (record sets, fields, columns) are referenced by their Croissant `@id`.

In [ ]:
# List all record sets with their @id and name
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Fallback: try to find record sets in the metadata json fields
    from mlcroissant._src.structure.graph import as_json
    meta_dict = as_json(metadata)
    if 'recordSet' in meta_dict:
        record_sets = meta_dict['recordSet']
    elif 'record_sets' in meta_dict:
        record_sets = meta_dict['record_sets']

if not record_sets:
    print("No record sets found in dataset metadata (schema contains no tables). Aborting.")
else:
    for rs in record_sets:
        # The record set might be a dict, or an object with .id and .name
        rs_id = getattr(rs, 'id', None) or rs.get('@id')
        rs_name = getattr(rs, 'name', None) or rs.get('name')
        print(f"Record set @id: {rs_id} - name: {rs_name}")

    # For demonstration, print the columns/fields of the first record set
    selected_record_set_id = getattr(record_sets[0], 'id', None) or record_sets[0].get('@id')
    print(f"\nColumns/fields for record set {selected_record_set_id}:")
    rs_obj = record_sets[0] if hasattr(record_sets[0], 'fields') or hasattr(record_sets[0], 'columns') else dataset._graph.from_id(selected_record_set_id)
    fields = []
    if hasattr(rs_obj, 'fields'):
        fields = rs_obj.fields
    elif hasattr(rs_obj, 'columns'):
        fields = rs_obj.columns
    elif isinstance(rs_obj, dict):
        if 'field' in rs_obj:
            fields = rs_obj['field']
        elif 'fields' in rs_obj:
            fields = rs_obj['fields']
        elif 'columns' in rs_obj:
            fields = rs_obj['columns']
    for field in fields:
        field_id = getattr(field, 'id', None) or field.get('@id')
        field_name = getattr(field, 'name', None) or field.get('name')
        print(f"  field/column @id: {field_id} - name: {field_name}")


## 3. Data Extraction

We load all available record sets (tables) from the dataset into pandas DataFrames, using each record set's `@id` for consistent referencing.

In [ ]:
# List all record sets (@id), extract as DataFrames
record_set_ids = []
record_sets_json = []
if hasattr(metadata, 'record_sets'):
    record_sets_json = metadata.record_sets
else:
    from mlcroissant._src.structure.graph import as_json
    meta_dict = as_json(metadata)
    if 'recordSet' in meta_dict:
        record_sets_json = meta_dict['recordSet']
    if 'record_sets' in meta_dict:
        record_sets_json = meta_dict['record_sets']

# get all the record set IDs
for rs in record_sets_json:
    if isinstance(rs, dict):
        rs_id = rs.get('@id')
    else:
        rs_id = getattr(rs, 'id', None)
        if rs_id is None:
            # try as dict
            rs_id = getattr(rs, '@id', None)
    if rs_id:
        record_set_ids.append(rs_id)

dfs = {}  # Store DataFrames by record set @id

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if not records:
            # If the table is empty, skip
            continue
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"Loaded {len(df)} records for record set: {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# If any record sets/tables were loaded, print first table's columns (by @id) and preview
if dfs:
    main_rs_id = list(dfs.keys())[0]
    print(f"\nColumns for primary record set: {main_rs_id}")
    print(list(dfs[main_rs_id].columns))
    display(dfs[main_rs_id].head())
else:
    print("No tabular data extracted. Please check the dataset schema.")


## 4. Exploratory Data Analysis (EDA)

Here, we demonstrate basic data analysis operations using fully qualified `@id` references.

Suppose among the loaded DataFrames/tables, we want to:
- Filter for patients above a certain age
- Normalize the "intervals between diagnoses" field
- Group data by sex/gender (or another categorical attribute), and compute means

All fields will be referenced by their full `@id` to follow Croissant conventions.

In [ ]:
# For demonstration, identify numeric and categorical fields in the main DataFrame
main_rs_id = list(dfs.keys())[0]  # Use first loaded record set
main_df = dfs[main_rs_id]

# Show column names for reference
print(f"Columns for {main_rs_id}:")
print(main_df.columns.tolist())

# Pick field id's for analysis: For this data, let's assume (based on medical dataset conventions) we choose:
# Numeric: age at 2nd CRC diagnosis (commonly present), interval_months for time between diagnoses if available
# Categorical: sex, anatomical_location

# Try common naming patterns for Croissant @id fields
numeric_field_candidates = [col for col in main_df.columns if 'age' in col.lower() or 'interval' in col.lower()]
group_field_candidates = [col for col in main_df.columns if 'sex' in col.lower() or 'gender' in col.lower() or 'anatomical' in col.lower()]

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # Prefer interval if present, else age
else:
    numeric_field_id = main_df.select_dtypes('number').columns[0]

if group_field_candidates:
    group_field_id = group_field_candidates[0]
else:
    group_field_id = main_df.select_dtypes('object').columns[0]

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Filtering records example: Keep rows with value > sample threshold
threshold = main_df[numeric_field_id].quantile(0.2) if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
filtered = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.1f} (N={len(filtered)})")
display(filtered[[numeric_field_id, group_field_id]].head())

# Normalize numeric field Z-score
filtered[f"{numeric_field_id}_normalized"] = (
    filtered[numeric_field_id] - filtered[numeric_field_id].mean()
) / filtered[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (mean=0, std=1):")
display(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical field, compute means of numeric fields
if group_field_id in filtered:
    grouped = filtered.groupby(group_field_id, observed=False)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped)


## 5. Visualization

Let's visualize the distribution of our numeric field, and compare group-wise means for selected categories (e.g., sex/gender or anatomical location). All axes/labels use full field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Barplot per group
if group_field_id in main_df and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.barplot(data=main_df, x=group_field_id, y=numeric_field_id, estimator='mean', ci=None)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and parse Croissant-annotated biomedical data using `mlcroissant`.
- Inspect record sets (tables) and fields via their canonical `@id`s.
- Perform basic EDA: filtering, normalization, and group-wise summary statistics, always referencing fields by `@id`.
- Visualize distributions and group effects for key clinical features.

This approach ensures that downstream processing pipelines and documentation are stable and robust to changes in field ordering or language, so long as `@id`s remain stable. For publication or sharing, always report fields and tables using their `@id` to guarantee reproducibility.

_For further analyses, consider multivariable grouping, outcome pivoting, or exporting processed tables by record set `@id`!_